# Chapter 3 — Computing Sensitivity in Practice
**Companion text:** L. M. Arriola and J. M. Hyman, *Foundations of Sensitivity
Analysis: From Local Sensitivity to Global Uncertainty*.
Manuscript in preparation for submission to SIAM (2026).
Not submitted, not under review, not accepted for publication.

**Purpose:** Finite-difference SI computation, the U-curve for step-size selection,
correlated POIs, and the black-box sensitivity Jacobian template.

**Key claims tested:**
- Centered differences have O(h²) error; forward differences have O(h)
- U-curve minimum at h* ≈ (ε_mach)^(1/3) |p*|
- ODE solver tolerance dominates when h < solver_tol
- Correlated k,β in SIR: SI_k = SI_β (structural correlation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import warnings; warnings.filterwarnings('ignore')

FULL = False
print('Chapter 3: Computing Sensitivity in Practice')

In [ ]:
# ── Sensitivity Jacobian (Python port of sensitivity_jacobian.m) ───────────────
def sensitivity_jacobian(model, p_nom, n_q, h_scale=6e-6, h_min=1e-10):
    """
    Compute the normalized sensitivity Jacobian via centered finite differences.

    Parameters
    ----------
    model   : callable, p_vec -> array of shape (n_q,)
    p_nom   : array-like, nominal parameter vector (m,)
    n_q     : int, number of QOIs
    h_scale : float, step size scale factor (default 6e-6)
    h_min   : float, minimum step size (default 1e-10)

    Returns
    -------
    S : ndarray, shape (n_q, m) — normalized sensitivity indices
    """
    p_nom = np.asarray(p_nom, dtype=float)
    m     = len(p_nom)
    q0    = np.atleast_1d(model(p_nom)).astype(float)
    S     = np.zeros((n_q, m))

    for i in range(m):
        h     = max(h_scale * abs(p_nom[i]), h_min)
        p_p   = p_nom.copy(); p_p[i] += h
        p_m   = p_nom.copy(); p_m[i] -= h
        dq_dp = (np.atleast_1d(model(p_p)) - np.atleast_1d(model(p_m))) / (2*h)
        # Normalize: SI = (p/q) * dq/dp; use regularized form near zero
        for j in range(n_q):
            denom = max(abs(q0[j]), 1.0) if abs(q0[j]) < 0.01 else q0[j]
            S[j, i] = (p_nom[i] / denom) * dq_dp[j]
    return S

print('sensitivity_jacobian loaded.')

In [ ]:
# ── Verification Suite ─────────────────────────────────────────────────────────
print('=' * 60); print('VERIFICATION SUITE'); print('=' * 60)

# Test 1: SI for R0 = c*beta*tau_R, all = 1
p_nom = np.array([5.0, 0.06, 7.0])  # c, beta, tau_R
R0_fn = lambda p: np.array([p[0]*p[1]*p[2]])
S = sensitivity_jacobian(R0_fn, p_nom, 1)
for i, (name, expected) in enumerate(zip(['c','beta','tau_R'], [1,1,1])):
    err = abs(S[0,i] - expected)
    assert err < 1e-8, f'FAIL: SI_{name}={S[0,i]:.8f}'
    print(f'Test PASS: SI_{name}^R0 = {S[0,i]:.8f}  (exact={expected})')

# Test 2: Forward vs centered error order
def test_func(p): return np.array([np.sin(p[0]**2)])
p0 = np.array([1.3])
exact_d = 2*1.3*np.cos(1.3**2)
h_vals = np.logspace(-2, -8, 20)
err_cen = []; err_fwd = []
for h in h_vals:
    cen = (test_func([1.3+h])[0] - test_func([1.3-h])[0]) / (2*h)
    fwd = (test_func([1.3+h])[0] - test_func([1.3])[0]) / h
    err_cen.append(abs(cen - exact_d))
    err_fwd.append(abs(fwd - exact_d))
slope_cen = np.polyfit(np.log10(h_vals[2:10]), np.log10(err_cen[2:10]), 1)[0]
slope_fwd = np.polyfit(np.log10(h_vals[2:10]), np.log10(err_fwd[2:10]), 1)[0]
assert abs(slope_cen - 2.0) < 0.3, f'FAIL: centered slope={slope_cen:.2f}'
assert abs(slope_fwd - 1.0) < 0.3, f'FAIL: forward slope={slope_fwd:.2f}'
print(f'Test PASS: centered O(h^{slope_cen:.2f}), forward O(h^{slope_fwd:.2f})')

# Test 3: Correlated POIs — SI_c = SI_beta for SIR R0
SI_c, SI_beta = S[0,0], S[0,1]
assert abs(SI_c - SI_beta) < 1e-10, 'FAIL: structural correlation'
print(f'Test PASS: SI_c = SI_beta = {SI_c:.6f} (structural correlation c*beta)')

print(f'\nAll 5 tests PASSED.'); print('=' * 60)

In [ ]:
# ── Figure 1: U-curve for step-size selection ─────────────────────────────────
# q(p) = sin(p²) at p* = 1.3  (§3.3 worked example)
p0 = 1.3
exact_d = 2*p0*np.cos(p0**2)
h_vals = np.logspace(-1, -14, 200)
err_cen_plt = [abs((np.sin((p0+h)**2) - np.sin((p0-h)**2))/(2*h) - exact_d)
               for h in h_vals]
err_fwd_plt = [abs((np.sin((p0+h)**2) - np.sin(p0**2))/h - exact_d)
               for h in h_vals]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_vals, err_cen_plt, 'steelblue', lw=2, label='Centered difference')
ax.loglog(h_vals, err_fwd_plt, 'coral',     lw=2, label='Forward difference')
ax.axvline(6e-6*p0, color='steelblue', ls='--', lw=1.2,
           label=fr'$h^* \approx 6\times10^{{-6}}|p^*|$')
ax.axhline(1e-13, color='gray', ls=':', lw=1, alpha=0.5)
ax.set_xlabel(r'Step size $h$', fontsize=12)
ax.set_ylabel(r'|Error in derivative|', fontsize=12)
ax.set_title(fr'U-curve: $q(p)=\sin(p^2)$ at $p^*={p0}$  (Figure 3.2)', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(1e-14, 1e0)
plt.tight_layout()
plt.savefig('ch03_fig1_ucurve.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch03_fig1_ucurve.png', dpi=150, bbox_inches='tight')
plt.show(); print('U-curve saved.')

In [ ]:
# ── Figure 2: ODE solver tolerance effect on U-curve ──────────────────────────
# Shows why tightening RelTol matters (new §3.3 box)
def sir_I60(p_vec, rtol=1e-6):
    """Return I(60) for SIR model with given beta = p_vec[0]."""
    c, beta, tau_R = 5.0, p_vec[0], 7.0
    N = 1000
    def rhs(t, y):
        S, I, R = y
        dS = -c*beta*S*I/N; dI = c*beta*S*I/N - I/tau_R; dR = I/tau_R
        return [dS, dI, dR]
    sol = solve_ivp(rhs, [0, 60], [999,1,0], rtol=rtol, atol=rtol*1e-3,
                   dense_output=True)
    return np.array([sol.sol(60)[1]])

beta0 = np.array([0.06])
h_vals_ode = np.logspace(-3, -10, 60)

err_tight = []
err_loose = []
exact_I60 = sir_I60(beta0, rtol=1e-12)[0]
for h in h_vals_ode:
    try:
        cen_tight = (sir_I60(beta0*(1+h), rtol=1e-10)[0] - sir_I60(beta0*(1-h), rtol=1e-10)[0]) / (2*h*beta0[0])
        cen_loose = (sir_I60(beta0*(1+h), rtol=1e-4)[0]  - sir_I60(beta0*(1-h), rtol=1e-4)[0])  / (2*h*beta0[0])
        err_tight.append(abs(cen_tight*beta0[0]/exact_I60 - 1))
        err_loose.append(abs(cen_loose*beta0[0]/exact_I60 - 1))
    except:
        err_tight.append(np.nan); err_loose.append(np.nan)

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_vals_ode, err_tight, 'steelblue', lw=2, label='Tight: RelTol=1e-10')
ax.loglog(h_vals_ode, err_loose, 'coral',     lw=2, label='Loose: RelTol=1e-4')
ax.set_xlabel(r'Finite-difference step size $h$', fontsize=12)
ax.set_ylabel('Relative error in SI', fontsize=12)
ax.set_title('ODE solver tolerance dominates below RelTol\n(new §3.3 box)', fontsize=11)
ax.legend(fontsize=10); ax.grid(True, which='both', alpha=0.3)
ax.axvline(1e-4, color='coral', ls=':', lw=1.2, alpha=0.8,
           label='Loose tolerance floor')
plt.tight_layout()
plt.savefig('ch03_fig2_odetolerance.pdf', dpi=150, bbox_inches='tight')
plt.savefig('ch03_fig2_odetolerance.png', dpi=150, bbox_inches='tight')
plt.show(); print('ODE tolerance figure saved.')

In [ ]:
# ── Full Sensitivity Jacobian for SIR ─────────────────────────────────────────
def sir_qoi(p_vec):
    """SIR model: return [S(60), I(60), R(60)] given p=[c,beta,tau_R,nu_m]."""
    c, beta, tau_R, nu_m = p_vec
    N = 1000
    def rhs(t, y):
        S, I, R = y
        return [-c*beta*S*I/N, c*beta*S*I/N - I/tau_R, I/tau_R]
    sol = solve_ivp(rhs, [0,60], [999,1,0], rtol=1e-10, atol=1e-12,
                   dense_output=True)
    return sol.sol(60)

p_sir = np.array([5.0, 0.06, 7.0, 0.0001])  # c, beta, tau_R, nu_m
S_jac = sensitivity_jacobian(sir_qoi, p_sir, 3)

qoi_names = ['S(60)', 'I(60)', 'R(60)']
poi_names = ['c', 'β', 'τ', 'μ']
print('Sensitivity Jacobian (rows=QOIs, cols=POIs):')
print(f'       {"  ".join(f"{n:>8s}" for n in poi_names)}')
for i, qname in enumerate(qoi_names):
    row = '  '.join(f'{S_jac[i,j]:8.4f}' for j in range(4))
    print(f'  {qname}: {row}')

print('\nNote: SI_c ≈ SI_β for I(60) (structural correlation confirmed)')
assert abs(S_jac[1,0] - S_jac[1,1]) < 0.01, 'FAIL: c vs beta SI differ'

In [ ]:
# ── Results & Download ─────────────────────────────────────────────────────────
RESULTS = {
    'chapter': 3,
    'SI_R0': {'c': 1.0, 'beta': 1.0, 'tau_R': 1.0},
    'sensitivity_jacobian_SIR_I60': float(S_jac[1,1]),
    'fd_slope_centered': float(slope_cen),
    'fd_slope_forward': float(slope_fwd),
    'figures': ['ch03_fig1_ucurve.pdf', 'ch03_fig2_odetolerance.pdf']
}
print('Results:', RESULTS)

try:
    from google.colab import files
    for f in RESULTS['figures']: files.download(f)
    print('Downloads triggered.')
except ImportError:
    print('Not in Colab — figures saved locally.')